# KS Fotoğraf → GLB
Ücretsiz Google Colab GPU üzerinde açık kaynak TripoSG ile GLB üretir. Üst menüden **Çalışma zamanı → Tümünü çalıştır** seçin. Kurulum otomatik yapılır; alttaki form açılınca fotoğrafı yükleyin.

In [ ]:
#@title KS 3D Üreticiyi Başlat { display-mode: "form" }
import subprocess, sys, urllib.request
from pathlib import Path
from IPython.display import display, HTML
display(HTML('<h3>KS 3D motoru hazırlanıyor…</h3><p>İlk kurulum ve model indirmesi 8–15 dakika sürebilir. Bu hücre Python 3.10 ortamını otomatik hazırlar.</p>'))
if subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    raise RuntimeError('GPU seçili değil. Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seçin.')
root = Path('/content/TripoSG')
if not root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSG.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
env = Path('/content/ks-py310')
if not env.exists():
    subprocess.run(['uv', 'venv', '--python', '3.10', str(env)], check=True)
python = str(env / 'bin/python')
req = root / 'requirements-ks.txt'
blocked = ('numpy', 'opencv-python', 'scikit-image', 'pymeshlab', 'diso')
req.write_text('\n'.join(line for line in (root / 'requirements.txt').read_text().splitlines() if not line.strip().lower().startswith(blocked)))
subprocess.run(['uv', 'pip', 'install', '--python', python, 'torch==2.4.1', 'torchvision==0.19.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
print('Temel 3D paketleri kuruluyor…')
subprocess.run(['uv', 'pip', 'install', '--python', python, 'numpy==1.26.4', 'scipy==1.11.4', 'scikit-image==0.22.0', 'opencv-python-headless==4.10.0.84', 'rembg==2.0.61', 'gradio>=5,<7', 'fast-simplification', 'setuptools', 'wheel', 'ninja', '-r', str(req)], check=True)
print('TripoSG CUDA yüzey çıkarıcı derleniyor…')
subprocess.run(['uv', 'pip', 'install', '--python', python, '--no-build-isolation', 'diso==0.1.4'], check=True)
subprocess.run([python, '-c', 'import torch, numpy, scipy, diso; print(\"Bağımlılık kontrolü tamam: Python 3.10 / NumPy \" + numpy.__version__)'], check=True)
script = Path('/content/ks_colab_producer.py')
urllib.request.urlretrieve('https://raw.githubusercontent.com/kenanseyhan-operasyon/ks-operations-center/ks-3d-studio/studio/colab_producer.py', script)
subprocess.run([python, '-u', str(script)], check=True)
